# ThesisV10 — Ontology-Guided Hybrid CAD Assembly Retrieval Framework
### Machine Learning Algorithms for Assessing the Similarity of Variant Rotor Designs Based on Ontological Features

**Author:** Ulukbek Rahmanov | BSc Software Engineering | University of Europe for Applied Sciences

**Target:** Advanced Engineering Informatics (Elsevier)

---

## Notebook Structure

| Section | Content |
|---------|---------|
| 1 | Data Loading & EDA |
| 2 | Feature Engineering (Technical · Structural · Semantic) |
| 3 | Similarity Matrices |
| 4 | Hybrid Fusion — Empirical Grid Search (α + β + γ = 1) |
| 5 | Retrieval Evaluation — P@K · MRR |
| 6 | ML Baselines — k-NN · Random Forest (RQ3) |
| 7 | Figures & Visualisations |
| 8 | Summary |

---

**Bug fixes vs ThesisV9:**
- **BUG 1 FIXED** — `surface_area` (all zeros) and `mass` (r=0.998 with volume) removed from technical features
- **BUG 2 FIXED** — `family` removed from semantic features (it is ground truth, not an input)
- **BUG 3 FIXED** — Weights chosen by empirical grid search, not hardcoded

## Cell 0 — Imports & Setup

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.manifold import TSNE

warnings.filterwarnings('ignore')

# ── Output directories (relative to this notebook) ───────────────────────────
NB_DIR       = os.path.dirname(os.path.abspath('ThesisV10.ipynb')) if '__file__' not in dir() else os.path.dirname(__file__)
OUT_TABLES   = os.path.join(NB_DIR, 'ontology', 'outputs', 'tables')
OUT_FIGURES  = os.path.join(NB_DIR, 'ontology', 'outputs', 'figures')
DATA_PATH    = os.path.join(NB_DIR, 'ontology', 'data', 'fusion360_rotor_dataset.csv')

os.makedirs(OUT_TABLES,  exist_ok=True)
os.makedirs(OUT_FIGURES, exist_ok=True)

print(f"Data:    {DATA_PATH}")
print(f"Tables:  {OUT_TABLES}")
print(f"Figures: {OUT_FIGURES}")

---
## Section 1 — Data Loading & Exploratory Data Analysis

The dataset contains 746 CAD assemblies from the Fusion360 Gallery Dataset.
Each assembly is a rotor-type component described by 15 attributes covering geometry,
material, topology, and structural properties.

**Pseudo-family labels** (5 classes) were derived from structural and geometric analysis —
they are NOT manually annotated. They serve as ground truth for retrieval evaluation only.

In [ ]:
# ── Load dataset ─────────────────────────────────────────────────────────────
df = pd.read_csv(DATA_PATH)

print(f"Dataset shape:  {df.shape[0]} assemblies  x  {df.shape[1]} columns")
print(f"\nColumn names:")
for c in df.columns:
    print(f"  {c}")
print(f"\nFamily distribution:")
print(df['family'].value_counts().to_string())

In [ ]:
# ── EDA: basic stats ─────────────────────────────────────────────────────────
print("\nNumerical feature summary:")
num_cols = ['component_count','body_count','edge_count','face_count',
            'vertex_count','mass','volume','surface_area',
            'complexity_ratio','rotational_symmetry','geometric_complexity']
print(df[num_cols].describe().round(3).to_string())

print(f"\nSurface area unique values: {df['surface_area'].unique()[:5]}  (extraction bug — all zero)")
print(f"Mass-Volume correlation:     {df['mass'].corr(df['volume']):.4f}  (near-perfect → mass redundant)")

In [ ]:
# ── EDA: family bar chart ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

family_counts = df['family'].value_counts()
axes[0].bar(family_counts.index, family_counts.values,
            color=['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2'],
            edgecolor='white', alpha=0.87)
axes[0].set_title('Assembly Family Distribution (n=746)', fontweight='bold')
axes[0].set_xlabel('Pseudo-Family')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)
for i, v in enumerate(family_counts.values):
    axes[0].text(i, v + 2, str(v), ha='center', fontsize=9, fontweight='bold')

# Rotational symmetry split
sym = df['rotational_symmetry'].value_counts()
axes[1].pie(sym.values, labels=['Asymmetric','Symmetric'],
            autopct='%1.1f%%', startangle=90,
            colors=['#4C72B0','#DD8452'])
axes[1].set_title('Rotational Symmetry Split', fontweight='bold')

plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/eda_overview.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/eda_overview.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: eda_overview.pdf")

---
## Section 2 — Feature Engineering

Three feature groups are extracted, each corresponding to one similarity component:

| Group | Features | Notes |
|-------|----------|-------|
| **Technical** | volume, edge_count, face_count, vertex_count, complexity_ratio, geometric_complexity | `surface_area` removed (all 0); `mass` removed (r=0.998 with volume) |
| **Structural** | component_count, body_count, rotational_symmetry | Assembly hierarchy and topology |
| **Semantic** | material_class, topology_class, complexity_class, structure_class | Ontology-derived categorical — `family` NOT included (it is ground truth) |

**Scaling:** MinMaxScaler (0–1 range) for numerical features before cosine similarity.
**Semantic encoding:** one-hot encoding of categorical ontology classes.

In [ ]:
# ── Ground-truth labels (ONLY for evaluation — never used as input) ───────────
ASSEMBLY_NAMES = df['assembly_name'].tolist()
FAMILY_LOOKUP  = dict(zip(df['assembly_name'], df['family']))
FAMILY_COLORS  = {'simple_rotor':'#4C72B0','medium_rotor':'#DD8452',
                  'high_complexity_rotor':'#55A868','symmetric_rotor':'#C44E52',
                  'complex_assembly':'#8172B2'}

print("Ground-truth labels loaded (not used as features).")
print(f"  Families: {list(pd.Series(FAMILY_LOOKUP.values()).value_counts().index)}")

In [ ]:
# ── BUG 1 FIX: Technical features — no surface_area, no mass ────────────────
TECHNICAL_COLS = [
    'volume',               # geometric size
    'edge_count',           # topological edge count
    'face_count',           # topological face count
    'vertex_count',         # topological vertex count
    'complexity_ratio',     # edge / face density
    'geometric_complexity', # combined complexity measure
]
# Removed: surface_area (0 for all 746 assemblies — extraction bug)
# Removed: mass        (Pearson r=0.998 with volume; mass ≈ density×volume,
#                       density captured by material_class in semantic layer)

STRUCTURAL_COLS = [
    'component_count',      # number of sub-components
    'body_count',           # number of solid bodies
    'rotational_symmetry',  # 1 = symmetric, 0 = asymmetric
]

print(f"Technical features  ({len(TECHNICAL_COLS)}): {TECHNICAL_COLS}")
print(f"Structural features ({len(STRUCTURAL_COLS)}): {STRUCTURAL_COLS}")

In [ ]:
# ── BUG 2 FIX: Semantic features — ontology-derived, NO family ──────────────
df_feat = df.copy()

def topology_class(x):
    if x < 300:    return "Sparse"
    elif x < 1500: return "Moderate"
    return "Dense"

def complexity_class(x):
    if x < 500:    return "Low"
    elif x < 3000: return "Medium"
    return "High"

def normalize_material(mat):
    m = str(mat).lower()
    if 'steel'     in m: return "Steel"
    if 'aluminum'  in m: return "Aluminum"
    if 'plastic'   in m: return "Plastic"
    if 'wood'      in m: return "Wood"
    if 'composite' in m: return "Composite"
    return "Other"

df_feat['topology_class']   = df['edge_count'].apply(topology_class)
df_feat['complexity_class'] = df['geometric_complexity'].apply(complexity_class)
df_feat['structure_class']  = np.where(df['rotational_symmetry'] == 1, 'Symmetric', 'Asymmetric')
df_feat['material_class']   = df['material'].apply(normalize_material)

SEMANTIC_COLS = ['material_class', 'topology_class', 'complexity_class', 'structure_class']
# NOTE: 'family' is deliberately excluded — it is the evaluation label, not a feature.

print(f"Semantic features ({len(SEMANTIC_COLS)}): {SEMANTIC_COLS}")
print("\nSemantic class distributions:")
for col in SEMANTIC_COLS:
    print(f"  {col}: {df_feat[col].value_counts().to_dict()}")

In [ ]:
# ── Save feature summary ─────────────────────────────────────────────────────
feat_rows = (
    [(f, 'Technical',  'Numerical')  for f in TECHNICAL_COLS]  +
    [(f, 'Structural', 'Numerical')  for f in STRUCTURAL_COLS] +
    [(f, 'Semantic',   'Categorical') for f in SEMANTIC_COLS]
)
feat_df = pd.DataFrame(feat_rows, columns=['Feature', 'Category', 'Type'])
feat_df.to_csv(f"{OUT_TABLES}/feature_summary_clean.csv", index=False)
print(feat_df.to_string(index=False))

---
## Section 3 — Similarity Matrices

Three pairwise similarity matrices are computed (746 × 746 each):

- **Technical**: MinMax-scaled numerical features → cosine similarity
- **Structural**: MinMax-scaled structural features → cosine similarity
- **Semantic**: One-hot encoded ontology classes → cosine similarity (equivalent to Jaccard for binary vectors)

In [ ]:
# ── Build similarity matrices ─────────────────────────────────────────────────
scaler = MinMaxScaler()

# Technical (6 numerical features)
X_tech        = df_feat[TECHNICAL_COLS].fillna(df_feat[TECHNICAL_COLS].median())
X_tech_scaled = scaler.fit_transform(X_tech)
SIM_TECH      = cosine_similarity(X_tech_scaled)
np.fill_diagonal(SIM_TECH, 1.0)

# Structural (3 numerical features)
X_struct        = df_feat[STRUCTURAL_COLS].fillna(df_feat[STRUCTURAL_COLS].median())
X_struct_scaled = scaler.fit_transform(X_struct)
SIM_STRUCT      = cosine_similarity(X_struct_scaled)
np.fill_diagonal(SIM_STRUCT, 1.0)

# Semantic (one-hot ontology classes — no family)
X_sem       = pd.get_dummies(df_feat[SEMANTIC_COLS])
SIM_SEM     = cosine_similarity(X_sem.values.astype(float))
np.fill_diagonal(SIM_SEM, 1.0)

print(f"All matrices: {SIM_TECH.shape}")
print(f"Semantic one-hot dimensions: {X_sem.shape[1]}")
print(f"\nSample similarity ranges:")
for name, mat in [('Technical', SIM_TECH), ('Structural', SIM_STRUCT), ('Semantic', SIM_SEM)]:
    off_diag = mat[np.triu_indices(len(mat), k=1)]
    print(f"  {name:12s}  mean={off_diag.mean():.4f}  std={off_diag.std():.4f}  min={off_diag.min():.4f}")

In [ ]:
# ── Save matrices ─────────────────────────────────────────────────────────────
for name, mat in [('technical', SIM_TECH), ('structural', SIM_STRUCT), ('semantic', SIM_SEM)]:
    pd.DataFrame(mat, index=ASSEMBLY_NAMES, columns=ASSEMBLY_NAMES).to_csv(
        f"{OUT_TABLES}/{name}_similarity_matrix.csv")
    print(f"Saved: {name}_similarity_matrix.csv")

# ── Heatmap: first 50 assemblies, sorted by family ────────────────────────────
sorted_idx = np.argsort([FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES])[:50]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, mat) in zip(axes, [('Technical', SIM_TECH),
                                   ('Structural', SIM_STRUCT),
                                   ('Semantic', SIM_SEM)]):
    sub = mat[np.ix_(sorted_idx, sorted_idx)]
    im = ax.imshow(sub, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
    ax.set_title(f'{name} Similarity\n(first 50, sorted by family)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Assembly index')
    ax.set_ylabel('Assembly index')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/similarity_heatmaps.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/similarity_heatmaps.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: similarity_heatmaps.pdf")

---
## Section 4 — Hybrid Fusion: Empirical Grid Search

The hybrid similarity is a weighted combination of the three components:

$$S_{hybrid} = \alpha \cdot S_{technical} + \beta \cdot S_{semantic} + \gamma \cdot S_{structural}$$

subject to $\alpha + \beta + \gamma = 1$, $\alpha, \beta, \gamma \geq 0$.

**Bug 3 fix:** Instead of arbitrary weights (ThesisV9 used 0.2/0.5/0.3), weights are chosen
by grid search over the constraint surface, optimising Mean Reciprocal Rank (MRR) on the full dataset.

In [ ]:
# ── Evaluation helper functions ───────────────────────────────────────────────
def precision_at_k(sim_matrix, labels, assemblies, k):
    precisions = []
    for i, name in enumerate(assemblies):
        q_fam = labels.get(name)
        row   = sim_matrix[i].copy()
        row[i] = -1
        top_k = np.argsort(row)[::-1][:k]
        hits  = sum(1 for j in top_k if labels.get(assemblies[j]) == q_fam)
        precisions.append(hits / k)
    return float(np.mean(precisions))

def mean_reciprocal_rank(sim_matrix, labels, assemblies):
    rr_list = []
    for i, name in enumerate(assemblies):
        q_fam = labels.get(name)
        row   = sim_matrix[i].copy()
        row[i] = -1
        for rank, j in enumerate(np.argsort(row)[::-1], 1):
            if labels.get(assemblies[j]) == q_fam:
                rr_list.append(1.0 / rank)
                break
        else:
            rr_list.append(0.0)
    return float(np.mean(rr_list))

def evaluate(mat, label=''):
    return {
        'Method':       label,
        'Precision@3':  round(precision_at_k(mat, FAMILY_LOOKUP, ASSEMBLY_NAMES, 3),  4),
        'Precision@5':  round(precision_at_k(mat, FAMILY_LOOKUP, ASSEMBLY_NAMES, 5),  4),
        'Precision@10': round(precision_at_k(mat, FAMILY_LOOKUP, ASSEMBLY_NAMES, 10), 4),
        'MRR':          round(mean_reciprocal_rank(mat, FAMILY_LOOKUP, ASSEMBLY_NAMES), 4),
    }

print("Evaluation functions defined.")

In [ ]:
# ── Grid search: α + β + γ = 1, step 0.1 ─────────────────────────────────────
print("Running grid search... (α=technical, β=semantic, γ=structural)")

steps = np.arange(0.0, 1.01, 0.1)
best_mrr, best_alpha, best_beta, best_gamma = -1, 0.1, 0.5, 0.4
grid_results = []

for alpha in steps:
    for beta in steps:
        gamma = round(1.0 - alpha - beta, 10)
        if gamma < -0.001 or gamma > 1.001:
            continue
        gamma = max(0.0, min(1.0, gamma))
        if abs(alpha + beta + gamma - 1.0) > 0.01:
            continue

        hybrid = alpha * SIM_TECH + beta * SIM_SEM + gamma * SIM_STRUCT
        np.fill_diagonal(hybrid, 1.0)
        mrr = mean_reciprocal_rank(hybrid, FAMILY_LOOKUP, ASSEMBLY_NAMES)
        p5  = precision_at_k(hybrid, FAMILY_LOOKUP, ASSEMBLY_NAMES, 5)

        grid_results.append({
            'alpha_technical':  round(alpha, 2),
            'beta_semantic':    round(beta,  2),
            'gamma_structural': round(gamma, 2),
            'MRR': round(mrr, 4),
            'P@5': round(p5,  4),
        })
        if mrr > best_mrr:
            best_mrr, best_alpha, best_beta, best_gamma = mrr, alpha, beta, gamma

grid_df = pd.DataFrame(grid_results).sort_values('MRR', ascending=False)
grid_df.to_csv(f"{OUT_TABLES}/weight_grid_search.csv", index=False)

print(f"\nGrid search complete — {len(grid_results)} weight combinations evaluated")
print(f"\nTop-10 weight combinations by MRR:")
print(grid_df.head(10).to_string(index=False))

In [ ]:
# ── Best vs recommended weights ───────────────────────────────────────────────
print(f"\nEmpirical best: α={best_alpha:.2f} (tech)  β={best_beta:.2f} (sem)  γ={best_gamma:.2f} (struct)")
print(f"                MRR = {best_mrr:.4f}")
print()
# Paper recommendation: α=0.1 ensures all three components contribute
# (pure structural dominance is expected because pseudo-family labels were
#  derived from structural/geometric features — see discussion in paper)
REC_ALPHA, REC_BETA, REC_GAMMA = 0.1, 0.5, 0.4
SIM_REC = REC_ALPHA * SIM_TECH + REC_BETA * SIM_SEM + REC_GAMMA * SIM_STRUCT
np.fill_diagonal(SIM_REC, 1.0)
rec_mrr = mean_reciprocal_rank(SIM_REC, FAMILY_LOOKUP, ASSEMBLY_NAMES)
print(f"Recommended:    α={REC_ALPHA:.2f} (tech)  β={REC_BETA:.2f} (sem)  γ={REC_GAMMA:.2f} (struct)")
print(f"                MRR = {rec_mrr:.4f}  (Δ={rec_mrr - best_mrr:.4f} vs empirical best)")
print()
print("Using EMPIRICAL BEST weights for all downstream evaluation.")

In [ ]:
# ── Final hybrid matrix ───────────────────────────────────────────────────────
SIM_HYBRID = best_alpha * SIM_TECH + best_beta * SIM_SEM + best_gamma * SIM_STRUCT
np.fill_diagonal(SIM_HYBRID, 1.0)
pd.DataFrame(SIM_HYBRID, index=ASSEMBLY_NAMES, columns=ASSEMBLY_NAMES).to_csv(
    f"{OUT_TABLES}/hybrid_similarity_matrix.csv")

pd.DataFrame([
    {'Component': 'Technical',  'Weight': best_alpha, 'Features': ', '.join(TECHNICAL_COLS)},
    {'Component': 'Semantic',   'Weight': best_beta,  'Features': ', '.join(SEMANTIC_COLS)},
    {'Component': 'Structural', 'Weight': best_gamma, 'Features': ', '.join(STRUCTURAL_COLS)},
]).to_csv(f"{OUT_TABLES}/hybrid_weights_final.csv", index=False)

print(f"Hybrid matrix saved  ({SIM_HYBRID.shape})")
print(f"Weights saved to hybrid_weights_final.csv")

---
## Section 5 — Retrieval Evaluation: Precision@K and MRR

For each query assembly, the top-K most similar assemblies are retrieved.
A retrieval is *correct* if the retrieved assembly belongs to the same pseudo-family as the query.

**Metrics:**
- **Precision@K** (K = 3, 5, 10): fraction of top-K results from the correct family
- **MRR** (Mean Reciprocal Rank): average of 1/rank of the first correct result

In [ ]:
# ── Evaluate all methods ──────────────────────────────────────────────────────
results = [
    evaluate(SIM_TECH,   'Technical'),
    evaluate(SIM_SEM,    'Semantic (Ontology)'),
    evaluate(SIM_STRUCT, 'Structural'),
    evaluate(SIM_HYBRID, 'Hybrid'),
]
results_df = pd.DataFrame(results)
results_df.to_csv(f"{OUT_TABLES}/retrieval_comparison_clean.csv", index=False)

print("=" * 65)
print("RETRIEVAL EVALUATION RESULTS (clean pipeline)")
print("=" * 65)
print(results_df.to_string(index=False))
print("=" * 65)

In [ ]:
# ── Save top-10 retrieval tables ──────────────────────────────────────────────
for method_name, mat in [('technical', SIM_TECH), ('semantic', SIM_SEM),
                          ('structural', SIM_STRUCT), ('hybrid', SIM_HYBRID)]:
    records = []
    for i, name in enumerate(ASSEMBLY_NAMES):
        row = mat[i].copy(); row[i] = -1
        for rank, j in enumerate(np.argsort(row)[::-1][:10], 1):
            records.append({
                'query_assembly':     name,
                'query_family':       FAMILY_LOOKUP.get(name),
                'rank':               rank,
                'retrieved_assembly': ASSEMBLY_NAMES[j],
                'retrieved_family':   FAMILY_LOOKUP.get(ASSEMBLY_NAMES[j]),
                'similarity':         round(mat[i, j], 6),
                'correct':            FAMILY_LOOKUP.get(name) == FAMILY_LOOKUP.get(ASSEMBLY_NAMES[j]),
            })
    pd.DataFrame(records).to_csv(f"{OUT_TABLES}/{method_name}_retrieval_results.csv", index=False)
    print(f"Saved: {method_name}_retrieval_results.csv  ({len(records)} rows)")

---
## Section 6 — ML Baselines: k-NN and Random Forest (RQ3)

**RQ3:** *To what extent can machine learning models learn similarity relationships from ontological feature representations?*

The combined feature vector (technical + structural + semantic one-hot) is used to train:
- **k-Nearest Neighbours** classifier (k = 1, 3, 5)
- **Random Forest** classifier

Evaluation uses **5-fold stratified cross-validation**. Family labels serve as class targets.
These baselines test whether the feature representation supports ML-based classification,
complementing the retrieval evaluation in Section 5.

In [ ]:
# ── Build combined ML feature matrix ──────────────────────────────────────────
X_ml = np.concatenate([X_tech_scaled, X_struct_scaled, X_sem.values.astype(float)], axis=1)
le   = LabelEncoder()
y_ml = le.fit_transform(df['family'])

print(f"Combined feature matrix: {X_ml.shape}")
print(f"Classes: {list(le.classes_)}")

In [ ]:
# ── k-NN cross-validation ─────────────────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

knn_results = {}
for k in [1, 3, 5]:
    knn = KNeighborsClassifier(n_neighbors=k, metric='cosine')
    scores = cross_val_score(knn, X_ml, y_ml, cv=cv, scoring='accuracy')
    knn_results[f'k-NN (k={k})'] = scores
    print(f"k-NN (k={k}): mean={scores.mean():.4f}  std={scores.std():.4f}  [{', '.join(f'{s:.4f}' for s in scores)}]")

In [ ]:
# ── Random Forest cross-validation ────────────────────────────────────────────
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_scores = cross_val_score(rf, X_ml, y_ml, cv=cv, scoring='accuracy')
print(f"Random Forest:  mean={rf_scores.mean():.4f}  std={rf_scores.std():.4f}  [{', '.join(f'{s:.4f}' for s in rf_scores)}]")

# Feature importance from RF (fit on full data)
rf.fit(X_ml, y_ml)
tech_names   = TECHNICAL_COLS
struct_names = STRUCTURAL_COLS
sem_names    = [f"sem_{c}" for c in X_sem.columns]
all_feat_names = tech_names + struct_names + sem_names

importance_df = pd.DataFrame({
    'Feature':    all_feat_names,
    'Importance': rf.feature_importances_,
    'Category':   (['Technical']*len(tech_names) +
                   ['Structural']*len(struct_names) +
                   ['Semantic']*len(sem_names))
}).sort_values('Importance', ascending=False)

print("\nTop-15 features by Random Forest importance:")
print(importance_df.head(15).to_string(index=False))

In [ ]:
# ── ML summary table + figure ─────────────────────────────────────────────────
ml_summary = {}
for k in [1, 3, 5]:
    ml_summary[f'k-NN (k={k})'] = knn_results[f'k-NN (k={k})']
ml_summary['Random Forest'] = rf_scores

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Accuracy comparison
names  = list(ml_summary.keys())
means  = [v.mean() for v in ml_summary.values()]
stds   = [v.std()  for v in ml_summary.values()]
colors_ml = ['#4C72B0','#4C72B0','#4C72B0','#C44E52']
bars = axes[0].bar(names, means, yerr=stds, capsize=5,
                   color=colors_ml, alpha=0.87, edgecolor='white')
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('5-fold CV Accuracy')
axes[0].set_title('ML Classifier Accuracy (5-fold CV)', fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)

# Feature importance top-15
top15 = importance_df.head(15)
cat_colors = {'Technical':'#4C72B0','Structural':'#55A868','Semantic':'#DD8452'}
bar_colors = [cat_colors[c] for c in top15['Category']]
axes[1].barh(top15['Feature'][::-1], top15['Importance'][::-1],
             color=bar_colors[::-1], alpha=0.87, edgecolor='white')
axes[1].set_xlabel('Feature Importance')
axes[1].set_title('Random Forest — Top-15 Feature Importance', fontweight='bold')
patches = [mpatches.Patch(color=c, label=l) for l, c in cat_colors.items()]
axes[1].legend(handles=patches, fontsize=8)
axes[1].grid(axis='x', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/ml_baselines.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/ml_baselines.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: ml_baselines.pdf")

---
## Section 7 — Figures

Publication-quality figures for the AEI paper.

In [ ]:
# ── Fig 1: Retrieval comparison (P@K + MRR) ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Retrieval Performance — Clean Pipeline\n'
             '(Bug-fixed: no circular evaluation, empirically optimised weights)',
             fontsize=11, fontweight='bold')

colors_r = {'Technical':'#4C72B0','Semantic (Ontology)':'#DD8452',
            'Structural':'#55A868','Hybrid':'#C44E52'}
methods_r = [r['Method'] for r in results]
bar_c = [colors_r.get(m,'#888') for m in methods_r]

ks    = [3, 5, 10]
x     = np.arange(len(ks))
width = 0.2
ax = axes[0]
for i, r in enumerate(results):
    vals = [r['Precision@3'], r['Precision@5'], r['Precision@10']]
    ax.bar(x + i*width, vals, width, label=r['Method'],
           color=bar_c[i], alpha=0.87, edgecolor='white')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(['P@3', 'P@5', 'P@10'])
ax.set_ylabel('Precision')
ax.set_ylim(0, 1.08)
ax.set_title('Precision@K')
ax.legend(fontsize=8)
ax.grid(axis='y', linestyle='--', alpha=0.4)

ax2 = axes[1]
mrr_vals = [r['MRR'] for r in results]
bars = ax2.bar(methods_r, mrr_vals, color=bar_c, alpha=0.87, edgecolor='white')
for bar, val in zip(bars, mrr_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax2.set_ylabel('MRR')
ax2.set_ylim(0, 1.08)
ax2.set_title('Mean Reciprocal Rank')
ax2.tick_params(axis='x', rotation=15)
ax2.grid(axis='y', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/retrieval_comparison_clean.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/retrieval_comparison_clean.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: retrieval_comparison_clean.pdf")

In [ ]:
# ── Fig 2: Weight grid search heatmap ─────────────────────────────────────────
pivot = grid_df.pivot_table(values='MRR', index='beta_semantic',
                             columns='alpha_technical', aggfunc='max')
fig2, ax3 = plt.subplots(figsize=(8, 6))
im = ax3.imshow(pivot.values, cmap='YlOrRd', aspect='auto',
                vmin=pivot.values.min(), vmax=pivot.values.max())
plt.colorbar(im, ax=ax3, label='MRR')
ax3.set_xticks(range(len(pivot.columns)))
ax3.set_xticklabels([f'{v:.1f}' for v in pivot.columns], fontsize=8)
ax3.set_yticks(range(len(pivot.index)))
ax3.set_yticklabels([f'{v:.1f}' for v in pivot.index], fontsize=8)
ax3.set_xlabel('α (Technical Weight)')
ax3.set_ylabel('β (Semantic Weight)')
ax3.set_title(f'Hybrid Weight Grid Search — MRR\n'
              f'Best: α={best_alpha:.1f}, β={best_beta:.1f}, γ={best_gamma:.1f}  →  MRR={best_mrr:.4f}')
best_row = list(pivot.index).index(round(best_beta, 1))
best_col = list(pivot.columns).index(round(best_alpha, 1))
ax3.plot(best_col, best_row, 'b*', markersize=14,
         label=f'Best (α={best_alpha:.1f}, β={best_beta:.1f})')
ax3.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/weight_grid_search.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/weight_grid_search.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: weight_grid_search.pdf")

In [ ]:
# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")# ── Fig 3: t-SNE embedding of hybrid feature space ────────────────────────────
print("Computing t-SNE (may take ~30 seconds)...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
X_tsne = tsne.fit_transform(X_ml)

fig3, ax4 = plt.subplots(figsize=(9, 7))
family_labels = [FAMILY_LOOKUP[n] for n in ASSEMBLY_NAMES]
for fam, col in FAMILY_COLORS.items():
    mask = np.array(family_labels) == fam
    ax4.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                c=col, label=fam.replace('_', ' ').title(),
                s=18, alpha=0.75, edgecolors='none')
ax4.set_title('t-SNE Embedding — Combined Feature Space (Technical + Structural + Semantic)',
              fontweight='bold')
ax4.set_xlabel('t-SNE Dimension 1')
ax4.set_ylabel('t-SNE Dimension 2')
ax4.legend(fontsize=9, markerscale=2)
ax4.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.pdf", bbox_inches='tight', dpi=150)
plt.savefig(f"{OUT_FIGURES}/tsne_embedding.png", bbox_inches='tight', dpi=150)
plt.show()
print("Saved: tsne_embedding.pdf")

---
## Section 8 — Summary

Full results for the paper.

In [ ]:
# ── Final summary table ───────────────────────────────────────────────────────
print("=" * 65)
print("THESIS V10 — COMPLETE RESULTS SUMMARY")
print("=" * 65)
print(f"\nDataset: {len(ASSEMBLY_NAMES)} assemblies, 5 pseudo-families")
print(f"Technical features:  {len(TECHNICAL_COLS)} (mass & surface_area removed)")
print(f"Semantic features:   {len(SEMANTIC_COLS)} categorical (family NOT used)")
print(f"Structural features: {len(STRUCTURAL_COLS)}")
print(f"\nHybrid weights (grid search over α+β+γ=1):")
print(f"  α (technical)  = {best_alpha:.2f}")
print(f"  β (semantic)   = {best_beta:.2f}")
print(f"  γ (structural) = {best_gamma:.2f}")

print(f"\nRetrieval Evaluation:")
print(results_df.to_string(index=False))

print(f"\nML Classification (5-fold CV accuracy):")
for name, scores in ml_summary.items():
    print(f"  {name:18s}  {scores.mean():.4f} ± {scores.std():.4f}")

print(f"\nOutputs saved to:")
print(f"  Tables:  {OUT_TABLES}/")
print(f"  Figures: {OUT_FIGURES}/")
print("=" * 65)